# 6. Signs above written beside their letter: retrain and update the demo

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chingkheinganba231005/meitei-mayek-word-recognition/blob/main/notebooks/6_signs_beside_demo.ipynb)

Handwriting often puts the signs that print places above a letter (aatap ꯥ, cheinap ꯩ,
nung ꯪ) beside it at the top, where yetnap ꯦ and otnap ꯣ stand. The model on Hugging Face
was trained with those signs only above their letter, so it sometimes misreads them there.
This notebook trains the same recipe again with **half of the training words writing ꯥ, ꯩ
and ꯪ beside their letter at the top** (the other half as in print), checks the new model on
synthetic words, and puts it on Hugging Face in place of the current one. Inap ꯤ (always
beside) and unap ꯨ (always below) are drawn as before. Training words are rendered on the
fly, as in notebook 3, so there is no data set to prepare.

1. **A look at the new training words** (a few minutes). Look before training.
2. **Test words with the signs beside** (a few minutes, once): the 5,000 validation and the
   5,000 test words again, with every sign above written beside its letter.
3. **Training**: about 2.5 hours on an A100; resumable if Colab disconnects.
4. **Checks on synthetic words**: the language model's weights chosen on validation, then
   the test words as printed and with the signs beside, for the new model and the current one.
5. **The demo** built from the new model.
6. **Upload** (`UPLOAD = True`): the current version on Hugging Face is first tagged
   `paper-v1` (the model the paper describes stays reachable there), then the new model and
   the demo page are uploaded.

Inputs on Drive, from notebooks 2-4: `glyphs/{train,val,test}.npz`,
`lexicon/{train,val,test}.tsv`, `synth/{val,test}.tar`, `lm/char_lm.pkl`, `weights/` (or
downloaded from Hugging Face), and the current demo model
`runs/round2_convnext_tummhcd_seed1/best.pt` with `results/phase2_val_round2_convnext_tummhcd_seed1.json`.
The upload needs a Hugging Face token with write access as a Colab secret called `HF_TOKEN`
(key icon on the left; switch on notebook access). Never paste the token into a cell.

In [ ]:
# Where things are. Change these to match your Drive.
WORK = "/content/drive/MyDrive/meitei-word-recognition"
REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
BRANCHES = ["main"]   # the branch to clone
HF_ID = "Chingkheinganba/handwritten-meitei-mayek-word-recognition"   # model repository and Space
HF_CHARS = "Chingkheinganba/handwritten-meitei-mayek-recognition"     # character-level weights, if not on Drive
RUN = "round3_convnext_tummhcd"
# the demo's recipe (ConvNeXt-T from the TUMMHCD character network, 10% scrambled words), with half of the
# training words writing the signs above (ꯥ ꯩ ꯪ) beside their letter at the top
RECIPE = {"encoder": "convnext_tiny", "init": "tummhcd", "scrambled": 0.1, "marks_beside": 0.5}
STEPS = 60000                              # 64 words each, as before
CURRENT = "round2_convnext_tummhcd_seed1"  # the model now on Hugging Face, for comparison
TAG = "paper-v1"                           # the Hugging Face tag that keeps the current version
UPLOAD = False                             # True to put the new model and demo on Hugging Face

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    if (r.stdout + r.stderr).strip():
        print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")

def stream(*args):
    """Runs a long command, showing its output as it comes; stops the notebook if it fails."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise SystemExit(f"exit code {p.returncode}: {' '.join(str(a) for a in args)[:200]}")

REPO_DIR = "/content/repo"
url = f"https://github.com/{REPO}.git"
if not os.path.exists(f"{REPO_DIR}/notebooks/6_signs_beside_demo.ipynb"):
    for branch in BRANCHES:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url, REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(f"{REPO_DIR}/notebooks/6_signs_beside_demo.ipynb"):
            break
    else:
        raise SystemExit("no branch in BRANCHES has the code. git: " + r.stderr)
else:  # a runtime left from an earlier session: bring the code up to date (its outputs are on Drive)
    branch = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
    run("git", "-C", REPO_DIR, "fetch", "-q", "--depth", "1", url, branch)
    run("git", "-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
run(sys.executable, "-m", "pip", "install", "-q", "timm", "huggingface_hub", "scipy", "onnx", "onnxruntime")

inputs = [f"glyphs/{s}.npz" for s in ("train", "val", "test")] + [f"lexicon/{s}.tsv" for s in ("train", "val", "test")]
inputs += ["synth/val.tar", "synth/test.tar", "lm/char_lm.pkl"]
needed = inputs + [f"runs/{CURRENT}/best.pt", f"results/phase2_val_{CURRENT}.json"]
missing = [f for f in needed if not os.path.exists(f"{WORK}/{f}")]
if missing:
    raise SystemExit("missing on Drive (from notebooks 2-4):\n  " + "\n  ".join(missing))
for f in inputs:
    os.makedirs(os.path.dirname(f"/content/{f}"), exist_ok=True)
    if not os.path.exists(f"/content/{f}"):
        shutil.copy(f"{WORK}/{f}", f"/content/{f}")   # local copies read faster than Drive
for d in ("results", "runs", "synth"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
WEIGHTS = "/content/weights"                  # the character-level weights: start of the encoder
if not os.path.exists(WEIGHTS):
    if os.path.exists(f"{WORK}/weights"):
        shutil.copytree(f"{WORK}/weights", WEIGHTS)
    else:
        from huggingface_hub import snapshot_download
        snapshot_download(HF_CHARS, local_dir=WEIGHTS)
        shutil.copytree(WEIGHTS, f"{WORK}/weights", dirs_exist_ok=True)
LM, SIZES = "/content/lm/char_lm.pkl", "results/glyph_sizes_tummhcd.json"
VAL, TEST = "/content/synth/val.tar", "/content/synth/test.tar"
VAL_BESIDE, TEST_BESIDE = "/content/synth/val_beside.tar", "/content/synth/test_beside.tar"
WORKERS = max((os.cpu_count() or 4) - 2, 2)   # processes rendering training words
LOOK = f"{WORK}/runs/{RUN}_look"              # the pictures of section 1
os.makedirs(LOOK, exist_ok=True)

def evaluate(name, data, out, tag, tune=False):
    """Run `name` reads a set: greedy, and with the language model at the weights chosen on validation
    (or chosen here, tune=True). Predictions and a sheet of misread words go to the run's folder."""
    choice = ["--tune"] if tune else ["--tuned", f"{WORK}/results/phase2_val_{name}.json"]
    run(sys.executable, "scripts/eval_recogniser.py", "--checkpoint", f"{WORK}/runs/{name}/best.pt",
        "--set", data, "--lm", LM, *choice, "--out", out,
        "--predictions", f"{WORK}/runs/{name}/{tag}_predictions.tsv", "--sheet", f"{WORK}/runs/{name}/{tag}_errors.png")

def scores(files):
    """A table of results files: CER and WER in %, greedy and with the language model."""
    import pandas as pd
    rows = []
    for label, f in files:
        if os.path.exists(f):
            r = json.load(open(f))
            g, w = r["greedy"], r["with_lm"]
            rows.append({"": label, "words": w["words"], "CER": round(100 * g["cer"], 2), "WER": round(100 * g["wer"], 1),
                         "CER LM": round(100 * w["cer"], 2), "WER LM": round(100 * w["wer"], 1)})
    return pd.DataFrame(rows)

run("git", "log", "-1", "--format=%h %s")
run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader")

## 1. A look at the new training words

Top: validation words with ꯥ, ꯩ or ꯪ, each drawn twice: left as print places the signs,
right beside their letter at the top. Below: the first 48 training words of the run (before
augmentation); about half of the words with a sign above have it beside. If something looks
wrong, stop here.

In [ ]:
import numpy as np
from IPython.display import Image as Show, display
from PIL import Image, ImageDraw, ImageFont
from mayek_words.glyphs import ASSETS, GlyphStore
from mayek_words.lexicon import Lexicon
from mayek_words.synth import Config, WordSynth, load_priors

ABOVE = "ꯥꯩꯪ"
val_store = GlyphStore.load("/content/glyphs/val.npz")
priors = load_priors(sizes=SIZES)
font = ImageFont.truetype(str(ASSETS / "NotoSansMeeteiMayek-Regular.ttf"), 20, layout_engine=ImageFont.Layout.RAQM)
lex = Lexicon.load("/content/lexicon/val.tsv")
rng, texts = np.random.default_rng(0), []
while len(texts) < 12:
    w = lex.sample(rng)
    if any(c in ABOVE for c in w) and w not in texts:
        texts.append(w)

def tile(img, text, h=72):
    im = Image.fromarray(img)
    im = im.resize((max(round(im.width * h / im.height), 1), h))
    t = Image.new("L", (im.width + 16, h + 28), 255)
    t.paste(im, (0, 0))
    ImageDraw.Draw(t).text((4, h + 2), text, font=font, fill=0)
    return t

pairs = [[tile(WordSynth(val_store, priors, Config(p_marks_beside=p)).render(w, np.random.default_rng([5, k])).image, w)
          for p in (0.0, 1.0)] for k, w in enumerate(texts)]
cw = max(t.width for pr in pairs for t in pr) + 12
ch = max(t.height for pr in pairs for t in pr) + 6
sheet = Image.new("L", (4 * cw, 6 * ch), 255)
for k, (a, b) in enumerate(pairs):
    r, c = divmod(k, 2)
    sheet.paste(a, (2 * c * cw, r * ch))
    sheet.paste(b, (2 * c * cw + cw, r * ch))
sheet.save(f"{LOOK}/print_and_beside.png")
display(Show(f"{LOOK}/print_and_beside.png"))

run(sys.executable, "scripts/render_words.py", "--glyphs", "/content/glyphs/train.npz", "--lexicon",
    "/content/lexicon/train.tsv", "--sizes", SIZES, "--n", 48, "--seed", 1000, "--scrambled", RECIPE["scrambled"],
    "--marks-beside", RECIPE["marks_beside"], "--sheet", f"{LOOK}/training_words.png")
display(Show(f"{LOOK}/training_words.png"))

## 2. Test words with the signs beside

The fixed validation and test sets again (the same words: the same seeds), with every sign
above written beside its letter. Made once and kept on Drive.

In [ ]:
for split, seed, local in (("val", 1, VAL_BESIDE), ("test", 2, TEST_BESIDE)):
    kept = f"{WORK}/synth/{split}_beside.tar"
    if not os.path.exists(kept):
        out = f"/content/synth/{split}_beside"
        shutil.rmtree(out, ignore_errors=True)
        run(sys.executable, "scripts/render_words.py", "--glyphs", f"/content/glyphs/{split}.npz", "--lexicon",
            f"/content/lexicon/{split}.tsv", "--sizes", SIZES, "--n", 5000, "--seed", seed, "--marks-beside", 1.0,
            "--out-dir", out)
        run("tar", "-cf", local, "-C", out, ".")
        shutil.copy(local, kept)
    elif not os.path.exists(local):
        shutil.copy(kept, local)
    print(f"{split}: {kept}")

## 3. Training

As for the current model (60,000 steps of 64 words, the best step on the validation words
kept), with half of the training words writing the signs above beside their letter. The run
saves its state every 1,000 steps to `WORK/runs/round3_convnext_tummhcd`; if Colab disconnects, run the first
two cells and this one again and it goes on from there.

In [ ]:
flags = [a for k, v in RECIPE.items() for a in ("--" + k.replace("_", "-"), v)]
stream(sys.executable, "-u", "scripts/train_recogniser.py", "--glyphs", "/content/glyphs/train.npz",
       "--lexicon", "/content/lexicon/train.tsv", "--sizes", SIZES, "--val", VAL,
       "--run-dir", f"{WORK}/runs/{RUN}", "--tummhcd-dir", WEIGHTS,
       "--results", f"{WORK}/results/phase2_train_{RUN}.json", "--steps", STEPS, "--workers", WORKERS, *flags)

## 4. Checks on synthetic words

The new model reads the validation words with the language model's weight (alpha) and bonus
per character (beta) chosen here, as before. Then the new and the current model read the test
words (TUMMHCD test characters) as printed and with every sign above beside, once each. The
new model should read the words with the signs beside far better than the current one, and
the printed words about as well.

In [ ]:
if not os.path.exists(f"{WORK}/runs/{RUN}/final.pt"):
    raise SystemExit(f"{RUN} is not trained yet: run section 3 first")
jobs = [(RUN, VAL, f"phase2_val_{RUN}.json", "val", True),
        (RUN, VAL_BESIDE, f"round3_val_beside_{RUN}.json", "val_beside", False),
        (RUN, TEST, f"phase2_test_{RUN}.json", "test", False),
        (RUN, TEST_BESIDE, f"round3_test_beside_{RUN}.json", "test_beside", False),
        (CURRENT, TEST, f"phase2_test_{CURRENT}.json", "test", False),
        (CURRENT, TEST_BESIDE, f"round3_test_beside_{CURRENT}.json", "test_beside", False)]
for name, data, out, tag, tune in jobs:
    if not os.path.exists(f"{WORK}/results/{out}"):
        evaluate(name, data, f"{WORK}/results/{out}", tag, tune=tune)
display(scores([(f"{n}: {label}", f"{WORK}/results/{f}") for n in (CURRENT, RUN)
                for label, f in (("test, as printed", f"phase2_test_{n}.json"),
                                 ("test, signs beside", f"round3_test_beside_{n}.json"))]))

## 5. The demo

The model repository and the static Space, built from the new model in `WORK/hf_round3_convnext_tummhcd`: the
network exported to ONNX (checked against PyTorch on validation words), the language model
for the browser, and the cards, which quote the checks of section 4 and say how the training
words place the signs above.

In [ ]:
HF_DIR = f"{WORK}/hf_{RUN}"
stream(sys.executable, "scripts/build_demo.py", "--runs", f"{WORK}/runs", "--results", f"{WORK}/results",
       "--candidates", RUN, "--lm", LM, "--check-set", VAL, "--model-dir", f"{HF_DIR}/model",
       "--space-dir", f"{HF_DIR}/space", "--hf-id", HF_ID, "--summary", f"{WORK}/results/demo_{RUN}.json")
for d in ("model", "space"):
    print(f"\n{d}:")
    for p in sorted(Path(f"{HF_DIR}/{d}").rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(HF_DIR)}  {p.stat().st_size / 1e6:.2f} MB")
print("\n" + Path(f"{HF_DIR}/model/README.md").read_text(encoding="utf-8"))

## 6. Upload

Set `UPLOAD = True` in the first cell, run the first two cells and this one. The model and
the Space as they are now are tagged `paper-v1` first (once), so the version the paper
describes stays at `huggingface.co/<id>/tree/paper-v1`; then the new files replace them. The
page loads the model from the repository, so reload it (a hard refresh) to try the new one.

In [ ]:
if not UPLOAD:
    print("UPLOAD is False: nothing is uploaded.")
else:
    from google.colab import userdata
    from huggingface_hub import HfApi
    try:
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        raise SystemExit("Add a Hugging Face token with write access as a Colab secret called HF_TOKEN (key icon "
                         "on the left) and allow this notebook to use it, then run this cell again.")
    HF_DIR = f"{WORK}/hf_{RUN}"
    if not os.path.exists(f"{HF_DIR}/model/web/model.onnx"):
        raise SystemExit("build the demo first (section 5)")
    api = HfApi(token=hf_token)
    for kind in ("model", "space"):
        tags = [t.name for t in api.list_repo_refs(HF_ID, repo_type=kind).tags]
        if TAG not in tags:
            api.create_tag(HF_ID, tag=TAG, repo_type=kind,
                           tag_message="The version described in the paper (every sign above in its printed place)")
            print(f"{kind}: the current version tagged {TAG}")
        else:
            print(f"{kind}: {TAG} already tagged")
    api.upload_folder(folder_path=f"{HF_DIR}/model", repo_id=HF_ID, repo_type="model",
                      commit_message=f"{RUN}: trained with the signs above also written beside their letter")
    api.upload_folder(folder_path=f"{HF_DIR}/space", repo_id=HF_ID, repo_type="space",
                      commit_message=f"Demo: {RUN}")
    print(f"model: https://huggingface.co/{HF_ID}")
    print(f"demo:  https://huggingface.co/spaces/{HF_ID}")
    print(f"the earlier version: https://huggingface.co/{HF_ID}/tree/{TAG}")